<a href="https://colab.research.google.com/github/jdasam/aat3020/blob/2026/notebooks/3_language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/jdasam/aat3020/blob/2026/notebooks/3_language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Language Modeling with RNN

In this notebook, we will build a **character-level language model** using a Recurrent Neural Network (RNN).

**Goal**: Train a model that can generate realistic-sounding names, one character at a time.

**What we'll cover:**
1. Data loading and exploration
2. Building a character vocabulary
3. Understanding RNN mechanics
4. Implementing a language model
5. Training the model
6. Generating new names

## Setup

Import the necessary libraries.

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

def read_txt(txt_path):
  with open(txt_path, 'r') as f:
    txt_string = f.readlines()
  return txt_string

## 1. Data Loading and Exploration

We'll use a dataset of names from Andrej Karpathy's `makemore` project.
- The dataset contains **32,033 names**
- Our goal: train a model to generate new names that sound similar

In [2]:
!wget -q "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"

txt_string = read_txt('names.txt')
names_list = [x.replace('\n', '') for x in txt_string]

print(f"Total names: {len(names_list)}")
print(f"First 10 names: {names_list[:10]}")

Total names: 32033
First 10 names: ['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


## 2. Building a Character Vocabulary

Unlike word-level models, a **character-level model** works with individual characters.

We need to:
1. Collect all unique characters (our **vocabulary**)
2. Add special tokens: `<pad>`, `<start>`, `<end>`
3. Map each character to an integer index (`char2idx`) and back (`idx2char`)

| Token | Purpose |
|---|---|
| `<pad>` | Fill shorter sequences to match batch length |
| `<start>` | Signal the beginning of a name |
| `<end>` | Signal the end of a name |

In [6]:
entire_chars = []
# TODO: iterate over names_list and collect all characters into entire_chars
for name in names_list:
  for char in name:
    entire_chars.append(char)
special_tokens = ['<pad>', '<start>', '<end>']
# TODO: build vocab as a sorted list of unique characters from entire_chars,
#       then prepend special_tokens
vocab = sorted(list(set(entire_chars)))
vocab = special_tokens + vocab
vocab

['<pad>',
 '<start>',
 '<end>',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [7]:

char2idx = {char: i for i, char in enumerate(vocab)}
idx2char = vocab

print(f"Vocabulary size: {len(vocab)}")
print(f"Vocabulary: {vocab}")
print(f"char2idx example: 'a' -> {char2idx['a']}, '<start>' -> {char2idx['<start>']}")

Vocabulary size: 29
Vocabulary: ['<pad>', '<start>', '<end>', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
char2idx example: 'a' -> 3, '<start>' -> 1


## 3. Recurrent Neural Networks (RNN)

An RNN processes sequential data by maintaining a **hidden state** that carries information from previous time steps.

**RNN Update Rule:**
$$h_t = \tanh(\mathbf{W}_{hh} h_{t-1} + \mathbf{W}_{xh} x_t + b)$$

- $x_t$: input vector at time step $t$
- $h_{t-1}$: hidden state from the previous step
- $h_t$: new hidden state (also the output at step $t$)
- $\mathbf{W}_{hh}$, $\mathbf{W}_{xh}$: learned weight matrices

Let's implement this step by step!

In [31]:
torch.manual_seed(0)
sequence_length = 7
input_dim, hidden_dim = 3, 5

# Weight matrices (using nn.Linear which computes W*x + b)
# TODO: define weight_hh as nn.Linear(hidden_dim, hidden_dim)
# TODO: define weight_xh as nn.Linear(input_dim, hidden_dim)
weight_hh = nn.Linear(hidden_dim,hidden_dim)
weight_xh = nn.Linear(input_dim,hidden_dim)

# Initial hidden state and input sequence
# TODO: define h0 as a zero vector of size hidden_dim
# TODO: define x as a random tensor of shape (sequence_length, input_dim)
h0 = torch.zeros(hidden_dim)
x = torch.randn(sequence_length,input_dim)

print(f"Input sequence shape: {x.shape}  -> (seq_len, input_dim)")
print(f"Initial hidden state: {h0}")

Input sequence shape: torch.Size([7, 3])  -> (seq_len, input_dim)
Initial hidden state: tensor([0., 0., 0., 0., 0.])


### Implementing a Single RNN Step

Given the previous hidden state $h_{t-1}$ and current input $x_t$, compute the new hidden state $h_t$.

In [32]:
def rnn_cell(weight_hh, weight_xh, prev_h, x_t):
  # TODO: return tanh(weight_hh(prev_h) + weight_xh(x_t))
  return torch.tanh(weight_hh(prev_h) + weight_xh(x_t))
  # pass


# Test with the first time step
h1 = rnn_cell(weight_hh, weight_xh, h0, x[0])
print(f"x[0]: {x[0]}")
print(f"h1:   {h1}")

x[0]: tensor([ 1.0554,  0.1778, -0.2303])
h1:   tensor([-0.3031,  0.4942, -0.3826, -0.1671, -0.0307], grad_fn=<TanhBackward0>)


### Running RNN Over a Full Sequence

We repeatedly apply the RNN cell, passing the hidden state from one step to the next.

In [33]:
# TODO: initialize output = [] and prev_h = h0
# TODO: loop over t in range(len(x)), apply rnn_cell,
#       append h to output, and update prev_h
output = []

prev_h = h0
for t in range(len(x)):
  x_t = x[t]
  h_t = rnn_cell(weight_hh, weight_xh, prev_h, x_t) # output[-1] or output[t-1]
  output.append(h_t)
  prev_h = h_t



output = torch.stack(output)
print(f"Output shape: {output.shape}  -> (seq_len, hidden_dim)")
output

Output shape: torch.Size([7, 5])  -> (seq_len, hidden_dim)


tensor([[-0.3031,  0.4942, -0.3826, -0.1671, -0.0307],
        [ 0.2949,  0.2907,  0.5566, -0.6004, -0.4537],
        [-0.0504, -0.8319,  0.6891, -0.0811, -0.9549],
        [-0.9035,  0.7153, -0.9110,  0.4101,  0.4610],
        [ 0.5157,  0.1567,  0.7691, -0.8519, -0.4661],
        [-0.6471,  0.9578, -0.5932, -0.2097,  0.4347],
        [ 0.2194,  0.3107,  0.5832, -0.7386, -0.3476]],
       grad_fn=<StackBackward0>)

### Using PyTorch's Built-in GRU

In practice, we use **GRU (Gated Recurrent Unit)** — an improved variant of RNN that better handles long-range dependencies via learned gates that control information flow.

PyTorch's `nn.GRU` runs the entire sequence at once (efficiently implemented in C++/CUDA).

In [35]:
# TODO: create gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, batch_first=True)
# TODO: add a batch dimension to x with unsqueeze(0), then run it through the GRU
rnn = nn.RNN(input_size = input_dim,hidden_size= hidden_dim, num_layers= 2, batch_first= True)
rnn_out, last_hidden = rnn(x.unsqueeze(0))

# x_batch = x.unsqueeze(1).repeat(1, 4, 1)
# print(x_batch.shape)

# for x in x_batch:
#   print(x.shape)

rnn_out
print(f"GRU output shape:      {rnn_out.shape}     -> (batch, seq_len, hidden_dim)")
print(f"Last hidden shape:     {last_hidden.shape}  -> (num_layers, batch, hidden_dim)")

GRU output shape:      torch.Size([1, 7, 5])     -> (batch, seq_len, hidden_dim)
Last hidden shape:     torch.Size([2, 1, 5])  -> (num_layers, batch, hidden_dim)


## 4. Dataset Class

We need to convert names into character-index sequences for training.

**Teacher Forcing**: During training, we always feed the *ground-truth* previous character as input (not the model's own prediction). This makes training stable.

For a name like `"emma"`:
- **Input**:  `[<start>, e, m, m, a]`
- **Target**: `[e, m, m, a, <end>]`

The model learns to predict the next character given all previous characters.

In [38]:
from torch.utils.data import Dataset

class NameSet(Dataset):
  def __init__(self, txt_fn):
    txt_string = read_txt(txt_fn)
    names_list = [x.replace("\n", '') for x in txt_string]
    self.data = names_list

    entire_chars = []
    # TODO: iterate over names_list and collect all characters into entire_chars
    for name in names_list:
      for char in name:
        entire_chars.append(char)
    special_tokens = ['<pad>', '<start>', '<end>']
    # TODO: build vocab as a sorted list of unique characters from entire_chars,
    #       then prepend special_tokens
    vocab = sorted(list(set(entire_chars)))
    self.vocab = special_tokens + vocab
    vocab
    # TODO: collect all characters from names_list into entire_chars
    # TODO: set self.vocab = special_tokens + sorted unique chars
    self.char2idx = {char: i for i, char in enumerate(self.vocab)}

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    # TODO: get name string from self.data[idx]
    name = self.data[idx]
    # TODO: convert each character to its index using self.char2idx
    name_in_idx = [self.char2idx[char] for char in name]

    # TODO: prepend <start> index and append <end> index
    name_in_idx = [self.char2idx['<start>']] + name_in_idx + [self.char2idx['<end>']]
    print(name_in_idx)
    # TODO: model_input = all tokens except the last
    model_input = name_in_idx[:-1]
    target_output = name_in_idx[1:]
    # TODO: target_output = all tokens except the first
    return model_input, target_output


dataset = NameSet('names.txt')
vocab_size = len(dataset.vocab)
print(f"Vocabulary: {dataset.vocab}")
print(f"Vocabulary size: {vocab_size}")
print(f"\nExample - 'emma' (index 0):")
print(f"  input  = {dataset[0][0]}")
print(f"  target = {dataset[0][1]}")

Vocabulary: ['<pad>', '<start>', '<end>', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Vocabulary size: 29

Example - 'emma' (index 0):
[1, 7, 15, 15, 3, 2]
  input  = [1, 7, 15, 15, 3]
[1, 7, 15, 15, 3, 2]
  target = [7, 15, 15, 3, 2]


## 5. Language Model Architecture

Our model takes a sequence of character indices and predicts the probability distribution over the **next character** at each position.

```
Input (character indices)
    ↓
Embedding Layer   →  converts integer indices to dense vectors
    ↓
GRU Layer         →  processes sequence, captures temporal context
    ↓
Linear Layer      →  maps hidden state to vocabulary size (logits)
    ↓
Softmax           →  converts logits to probability distribution
```

In [ ]:
class LanguageModel(nn.Module):
  def __init__(self, vocab_size, embedding_dim=16):
    super().__init__()
    # TODO: define self.emb as nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
    # TODO: define self.rnn as nn.GRU(input_size=embedding_dim, hidden_size=2*embedding_dim,
    #                                  num_layers=3, batch_first=True)
    # TODO: define self.proj as nn.Linear(in_features=2*embedding_dim, out_features=vocab_size)

  def forward(self, x):
    # TODO: pass x through self.emb
    # TODO: pass through self.rnn, unpack (output, last_hidden)
    # TODO: pass through self.proj
    # TODO: apply softmax along last dimension and return
    pass


# Test the model with a single example (no batch dimension)
model = LanguageModel(vocab_size)
x_in, y_out = dataset[0]
x_tensor = torch.tensor(x_in)
print(f"Input tokens: {x_tensor}")
print(f"Output shape: {model(x_tensor).shape}  -> (seq_len, vocab_size)")

## 6. Collate Function (Padding)

Names have different lengths, but tensors in a batch must have the same shape.
We **pad** shorter sequences with the `<pad>` token (index `0`) on the right.

```
"ava"      -> [<start>, a, v, a, <end>, <pad>, <pad>]
"emma"     -> [<start>, e, m, m, a, <end>, <pad>]
"olivia"   -> [<start>, o, l, i, v, i, a, <end>]
```

The loss function will ignore `<pad>` tokens so they don't affect learning.

In [ ]:
from torch.utils.data import DataLoader

def collate(raw_batch):
  x = [item[0] for item in raw_batch]
  y = [item[1] for item in raw_batch]

  # TODO: find max_len = max sequence length in the batch
  # TODO: pad each sequence in x and y with 0s to max_len,
  #       collect results into pad_x and pad_y
  # TODO: return torch.tensor(pad_x), torch.tensor(pad_y)


# Test
dataloader = DataLoader(dataset, batch_size=8, collate_fn=collate)
batch_x, batch_y = next(iter(dataloader))
print(f"Batch input shape:  {batch_x.shape}  -> (batch_size, max_seq_len)")
print(f"Batch target shape: {batch_y.shape}")
print(f"\nFirst 3 input sequences:")
print(batch_x[:3])

## 7. Training

### Loss Function: Negative Log-Likelihood (NLL)

For each position in the sequence, the model outputs a probability distribution over the vocabulary.
We want to **maximize** the probability of the correct next character, which is equivalent to **minimizing** its negative log probability:

$$\mathcal{L} = -\frac{1}{N} \sum_{t} \log p(\text{correct char at } t)$$

We skip the loss for `<pad>` tokens (index 0) since they are not real characters.

In [ ]:
def get_loss(prediction, y):
  batch_size, seq_length, vocab_size = prediction.shape
  prediction = prediction.reshape(batch_size * seq_length, vocab_size)
  y = y.reshape(-1)

  # Get the predicted probability for each correct character
  prob_of_correct_char = prediction[torch.arange(len(y)), y]

  # TODO: filter out padding tokens (where y == 0)
  # TODO: compute and return mean NLL: -log(prob + 1e-8)


# Quick test
x_test, y_test = dataset[0]
x_test = torch.tensor(x_test)
y_test = torch.tensor(y_test)
pred_test = model(x_test).unsqueeze(0)   # add batch dim
y_test_batch = y_test.unsqueeze(0)
print(f"Loss on single example: {get_loss(pred_test, y_test_batch):.4f}")
print(f"(Random baseline would be ~log({vocab_size}) = {torch.log(torch.tensor(float(vocab_size))):.2f})")

### Training Setup

Split the dataset into training (90%) and validation (10%) sets, then create data loaders.

In [ ]:
train_size = int(len(dataset) * 0.9)
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = LanguageModel(vocab_size, embedding_dim=32)
model.to(device)
optimizer = torch.optim.Adam(model.parameters())

train_loader = DataLoader(train_dataset, batch_size=64, collate_fn=collate, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, collate_fn=collate)

print(f"Training on: {device}")
print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

### Training Loop

For each epoch, we iterate over all training batches, compute the loss, and update the model parameters via backpropagation.

In [ ]:
loss_record = []
n_epoch = 10

for epoch in range(n_epoch):
  model.train()
  for batch in tqdm(train_loader, leave=False):
    x, y = batch
    pred = model(x.to(device))
    loss = get_loss(pred, y.to(device))
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    loss_record.append(loss.item())

  print(f"Epoch {epoch+1}/{n_epoch} - Loss: {loss.item():.4f}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(loss_record)
plt.title("Training Loss")
plt.xlabel("Step")
plt.ylabel("NLL Loss")
plt.show()

## 8. Text Generation (Inference)

Now let's use the trained model to **generate new names** character by character.

**Inference is different from training:**
- During training: we feed the *ground-truth* previous character (teacher forcing)
- During inference: we feed the model's *own previous output*

**Generation process:**
1. Start with the `<start>` token
2. Get the model's probability distribution over the next character
3. **Sample** a character from that distribution using `torch.multinomial`
4. Feed the sampled character back as input
5. Repeat until `<end>` token is sampled or max length is reached

Note: we reuse the **hidden state** across steps to preserve context, feeding only one token at a time.

In [ ]:
torch.set_printoptions(sci_mode=False)
model.to('cpu')
model.eval()

def generate_name(model, dataset, max_len=20):
  # TODO: initialize in_token = [<start> index], last_hidden = None

  # TODO: loop up to max_len times:
  #   - take the last token in in_token, shape it as (1, 1) with unsqueeze(0)
  #   - embed it with model.emb
  #   - run through model.rnn, passing and updating last_hidden
  #   - run through model.proj, apply softmax to get prob over vocab
  #   - sample next token with torch.multinomial(prob, num_samples=1)
  #   - if sampled_token is 0, 1, or 2 (<pad>/<start>/<end>): break
  #   - otherwise append sampled_token to in_token

  # TODO: return the joined character string (skip the first <start> token)
  pass


print("Generated names:")
for _ in range(10):
  print(" ", generate_name(model, dataset))